# Decision Graph — Mock Conversation Demo

This notebook simulates a realistic two-turn design session **without** running the live LangGraph agent.
It shows exactly what the decision graph looks like and what the API returns (`{nodes, edges, head}`).

## Simulated session

| Turn | User message | Agent actions |
|------|-------------|---------------|
| 1 | Place an L-shaped building on the south side | read_site → generate_shape → optimize_view (3 Pareto options) → user picks option 2 |
| 2 | Add a T-shaped building near north | analyze_remaining → generate_shape → optimize_two_building (4 Pareto options) → user picks option 1 |
| 3 (backtrack) | Actually I prefer option 3, more view for building 2 | user re-selects option 3 from turn 2's Pareto |

The graph will show **linear sequences** for action chains and **branches** for Pareto alternatives,  
plus the **backtrack fork** when the user changes their mind.

In [ ]:
import sys, os
from pathlib import Path
from collections import defaultdict
import plotly.graph_objects as go

# --- Shared runtime module (single source of truth) -----------------------
# The decision-graph logic this notebook validates is exposed by
# connection.notebook_logic.decision_graph_runtime, which the frontend decision
# tree also drives via /api/decision-graph/*. It re-exports the SAME DecisionGraph
# implementation (connection.decision_graph) — one source of truth everywhere.
# Add team_04/PY to sys.path so the connection package imports.
_here = Path.cwd().resolve()
TEAM_ROOT = next(
    (p for p in (_here, _here.parent, _here / "team_04", _here.parent / "team_04") if (p / "agent").exists()),
    None,
)
PY_ROOT = str(TEAM_ROOT / "PY") if TEAM_ROOT else os.path.abspath("../PY")
if PY_ROOT not in sys.path:
    sys.path.insert(0, PY_ROOT)

from connection.decision_graph import (
    DecisionGraph,
    make_intent_node, make_action_node,
    make_branch_nodes, make_state_node,
)
# Higher-level helpers the frontend/API use (new_graph, branch_options, select_path,
# backtrack, to_frontend_payload) — same graph, convenience wrappers:
from connection.notebook_logic import decision_graph_runtime as dg_runtime

print('DecisionGraph imported OK (via connection.notebook_logic source of truth)')


## Build the mock conversation graph

In [3]:
g = DecisionGraph()

# -----------------------------------------------------------------------
# TURN 1 — Place L-shaped building
# -----------------------------------------------------------------------
i1 = make_intent_node(g, 'Place an L-shaped building on the south side of the site')

a1 = make_action_node(g, 'read_site',
                      '{site_boundary: [[0,0],[100,0],[100,100],[0,100]]}', i1)
a2 = make_action_node(g, 'generate_building_boundary',
                      '{building_type: L, area: 675, requested_position: [50,15]}', a1)
a3 = make_action_node(g, 'optimize_view_placement',
                      '{building_type: L, attractors: [south_street], pop_size: 50}', a2)

pareto_1 = [
    {'rank': 1, 'option_id': 't1_opt1', 'combined_score': 0.820,
     'unblocked_view_score': 0.85, 'attractor_view_score': 0.75,
     'rotation_degrees': 0,  'centroid_xy': [28, 18], 'boundary': [], 'outside_area_sqm': 0},
    {'rank': 2, 'option_id': 't1_opt2', 'combined_score': 0.791,
     'unblocked_view_score': 0.80, 'attractor_view_score': 0.78,
     'rotation_degrees': 90, 'centroid_xy': [35, 22], 'boundary': [], 'outside_area_sqm': 0},
    {'rank': 3, 'option_id': 't1_opt3', 'combined_score': 0.765,
     'unblocked_view_score': 0.76, 'attractor_view_score': 0.77,
     'rotation_degrees': 45, 'centroid_xy': [22, 25], 'boundary': [], 'outside_area_sqm': 0},
]
b1 = make_branch_nodes(g, a3, pareto_1)

# User picks option 2
opt2_id = [n['node_id'] for n in g.children_of(b1) if 'Option 2' in n['label']][0]
g.select_node(opt2_id)
sel1 = g.add_node('select', 'Selected: Option 2 (rotation=90°)',
                   parent_id=opt2_id,
                   payload={'selected_option_id': 't1_opt2', 'reason': 'Best attractor score'})

print(f'Turn 1 complete — graph has {g.node_count()} nodes, head={g.current_head()[:8]}...')

# -----------------------------------------------------------------------
# TURN 2 — Add T-shaped building
# -----------------------------------------------------------------------
i2 = make_intent_node(g, 'Add a T-shaped building near the north edge facing south')

a4 = make_action_node(g, 'analyze_remaining_positions',
                      '{placed_building_id: bld_001, site_boundary: ...}', i2)
a5 = make_action_node(g, 'generate_building_boundary',
                      '{building_type: T, area: 600, requested_position: [55, 75]}', a4)
a6 = make_action_node(g, 'optimize_two_building_placement',
                      '{btype_1: L, btype_2: T, min_separation: 6, pop_size: 60}', a5)

pareto_2 = [
    {'rank': 1, 'option_id': 't2_opt1', 'combined_score': 0.796,
     'building_1': {'combined_score': 0.81, 'unblocked_view_score': 0.83},
     'building_2': {'combined_score': 0.76, 'unblocked_view_score': 0.74},
     'clearance_between_buildings_m': 8.2, 'boundary': []},
    {'rank': 2, 'option_id': 't2_opt2', 'combined_score': 0.783,
     'building_1': {'combined_score': 0.79, 'unblocked_view_score': 0.81},
     'building_2': {'combined_score': 0.78, 'unblocked_view_score': 0.79},
     'clearance_between_buildings_m': 6.5, 'boundary': []},
    {'rank': 3, 'option_id': 't2_opt3', 'combined_score': 0.771,
     'building_1': {'combined_score': 0.71, 'unblocked_view_score': 0.72},
     'building_2': {'combined_score': 0.85, 'unblocked_view_score': 0.88},
     'clearance_between_buildings_m': 9.1, 'boundary': []},
    {'rank': 4, 'option_id': 't2_opt4', 'combined_score': 0.748,
     'building_1': {'combined_score': 0.68, 'unblocked_view_score': 0.71},
     'building_2': {'combined_score': 0.82, 'unblocked_view_score': 0.85},
     'clearance_between_buildings_m': 11.3, 'boundary': []},
]
b2 = make_branch_nodes(g, a6, pareto_2)

# User picks option 1
opt1_t2_id = [n['node_id'] for n in g.children_of(b2) if 'Option 1' in n['label']][0]
g.select_node(opt1_t2_id)
sel2 = g.add_node('select', 'Selected: Option 1 (best avg score)',
                   parent_id=opt1_t2_id,
                   payload={'selected_option_id': 't2_opt1', 'reason': 'Best combined average'})

print(f'Turn 2 complete — graph has {g.node_count()} nodes')

# -----------------------------------------------------------------------
# TURN 3 — Backtrack to option 3
# -----------------------------------------------------------------------
# User changes mind — prefers option 3 (better B2 view even if B1 worse)
opt3_t2_id = [n['node_id'] for n in g.children_of(b2) if 'Option 3' in n['label']][0]
g.select_node(opt3_t2_id)   # deselects option 1, selects option 3

sel3 = g.add_node('select', 'Re-selected: Option 3 (B2 view priority)',
                   parent_id=opt3_t2_id,
                   payload={
                       'selected_option_id': 't2_opt3',
                       'reason': 'Higher view score for Building 2 (0.88 vs 0.74)',
                       'backtrack': True,
                   })

print(f'Turn 3 (backtrack) complete — graph has {g.node_count()} nodes')
print(f'Head: {g.current_head()[:8]}...')

# Confirm branch 2 child states
print('\nBranch 2 children after backtrack:')
for c in g.children_of(b2):
    print(f'  {c["label"]:35s} is_selected={c["is_selected"]}')

Turn 1 complete — graph has 9 nodes, head=a1dbf780...
Turn 2 complete — graph has 19 nodes
Turn 3 (backtrack) complete — graph has 20 nodes
Head: de2480d7...

Branch 2 children after backtrack:
  Option 1 (score=0.796)              is_selected=False
  Option 2 (score=0.783)              is_selected=False
  Option 3 (score=0.771)              is_selected=True
  Option 4 (score=0.748)              is_selected=False


## Selected path trace

Walk from head back to root following `parent_id` — this is the "active design history".

In [ ]:
def selected_path(graph):
    """Walk from head to root, return ordered list root → head."""
    path = []
    node_id = graph.current_head()
    while node_id:
        node = graph.get_node(node_id)
        if node is None:
            break
        path.append(node)
        node_id = node.get('parent_id')
    return list(reversed(path))

path = selected_path(g)
print(f'Active path ({len(path)} nodes, root → head):\n')
for i, n in enumerate(path):
    TYPE_ICONS = {'intent': 'USER', 'action': 'TOOL', 'branch': 'FORK',
                  'select': ' OK ', 'state': 'BLDG'}
    icon = TYPE_ICONS.get(n['type'], '    ')
    prefix = '  ' * min(i, 6)
    print(f'{prefix}[{icon}] {n["label"]}')

Active path (14 nodes, root → head):

[USER] User: Place an L-shaped building on the south side of the site
  [TOOL] Tool: read_site
    [TOOL] Tool: generate_building_boundary
      [TOOL] Tool: optimize_view_placement
        [FORK] Pareto alternatives (3 options)
          [BLDG] Option 2 (score=0.791)
            [ OK ] Selected: Option 2 (rotation=90°)
            [USER] User: Add a T-shaped building near the north edge facing south
            [TOOL] Tool: analyze_remaining_positions
            [TOOL] Tool: generate_building_boundary
            [TOOL] Tool: optimize_two_building_placement
            [FORK] Pareto alternatives (4 options)
            [BLDG] Option 3 (score=0.771)
            [ OK ] Re-selected: Option 3 (B2 view priority)


## API output — what React Flow / D3 receives

`GET /sessions/{id}/decisions` returns `{nodes, edges, head}`.
Edges already have `{id, source, target}` — React Flow needs no transformation.

In [5]:
import json
d = g.to_dict()
print(f'nodes: {len(d["nodes"])}')
print(f'edges: {len(d["edges"])}')
print(f'head:  {d["head"][:8]}...')
print()

# Show node type distribution
from collections import Counter
counts = Counter(n['type'] for n in d['nodes'])
for t, c in sorted(counts.items()):
    print(f'  {t:8s}: {c}')

print('\nFirst edge sample:', json.dumps(d['edges'][0], indent=2))

nodes: 20
edges: 19
head:  de2480d7...

  action  : 6
  branch  : 2
  intent  : 2
  select  : 3
  state   : 7

First edge sample: {
  "id": "e48b8562-6069-40ae-a168-5b87f67f0fbf-c6b4e2af-ddc6-44db-9c35-5ce4f4f5211a",
  "source": "e48b8562-6069-40ae-a168-5b87f67f0fbf",
  "target": "c6b4e2af-ddc6-44db-9c35-5ce4f4f5211a"
}


## Plotly DAG visualisation

Hierarchical layout:
- **Y axis** = depth level (top = root, down = later decisions)
- **X axis** = horizontal spread within each level
- **Thick solid edge** = selected path (active design history)
- **Thin dashed edge** = unselected branch

Node colours:
- Blue = user intent
- Orange = tool/action
- Purple = Pareto branch point
- Green = selected option
- Teal = building state / option

In [6]:
def compute_dag_layout(graph_dict):
    """Compute (x, y) positions for a DAG using a simple recursive subtree-width algorithm."""
    nodes = {n['node_id']: n for n in graph_dict['nodes']}
    children_map = defaultdict(list)
    for e in graph_dict['edges']:
        children_map[e['source']].append(e['target'])

    # Find roots (no parent)
    all_children = {e['target'] for e in graph_dict['edges']}
    roots = [n['node_id'] for n in graph_dict['nodes'] if n['node_id'] not in all_children]

    pos = {}
    x_counter = [0]          # mutable counter for leaf x positions

    def place(node_id, depth):
        kids = children_map[node_id]
        if not kids:
            x = x_counter[0]
            x_counter[0] += 1
            pos[node_id] = (x, -depth)
            return x
        child_xs = [place(c, depth + 1) for c in kids]
        x = sum(child_xs) / len(child_xs)
        pos[node_id] = (x, -depth)
        return x

    for r in roots:
        place(r, 0)

    return pos


TYPE_COLORS = {
    'intent':  '#2980b9',   # blue
    'action':  '#e67e22',   # orange
    'branch':  '#8e44ad',   # purple
    'select':  '#27ae60',   # green
    'state':   '#16a085',   # teal
}
TYPE_SYMBOLS = {
    'intent': 'diamond',
    'action': 'square',
    'branch': 'star',
    'select': 'circle',
    'state':  'hexagon',
}
TYPE_SIZES = {
    'intent': 28, 'action': 20, 'branch': 34, 'select': 24, 'state': 22,
}


def build_plotly_dag(graph, title='Decision Graph'):
    graph_dict = graph.to_dict()
    pos = compute_dag_layout(graph_dict)
    nodes_map = {n['node_id']: n for n in graph_dict['nodes']}
    selected_ids = {n['node_id'] for n in graph_dict['nodes'] if n['is_selected']}
    head_id = graph_dict['head']

    # Build selected-path set (walk from head to root)
    active_path_ids = set()
    nid = head_id
    while nid:
        active_path_ids.add(nid)
        nid = nodes_map[nid].get('parent_id')

    traces = []

    # --- Edges ---
    for e in graph_dict['edges']:
        src_id, tgt_id = e['source'], e['target']
        if src_id not in pos or tgt_id not in pos:
            continue
        x0, y0 = pos[src_id]
        x1, y1 = pos[tgt_id]
        on_active = src_id in active_path_ids and tgt_id in active_path_ids
        color = '#1a252f' if on_active else '#bdc3c7'
        width = 3.0      if on_active else 1.0
        dash  = 'solid'  if on_active else 'dot'
        traces.append(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None],
            mode='lines',
            line=dict(color=color, width=width, dash=dash),
            hoverinfo='none', showlegend=False,
        ))

    # --- Nodes (one trace per type for legend) ---
    by_type = defaultdict(list)
    for n in graph_dict['nodes']:
        if n['node_id'] in pos:
            by_type[n['type']].append(n)

    for ntype, ns in by_type.items():
        xs = [pos[n['node_id']][0] for n in ns]
        ys = [pos[n['node_id']][1] for n in ns]
        labels = [n['label'][:30] + ('…' if len(n['label']) > 30 else '') for n in ns]
        hover = [
            f"<b>{n['type'].upper()}</b><br>"
            f"{n['label']}<br>"
            f"selected={n['is_selected']}<br>"
            f"id={n['node_id'][:8]}"
            for n in ns
        ]
        color = TYPE_COLORS.get(ntype, '#7f8c8d')
        symbol = TYPE_SYMBOLS.get(ntype, 'circle')
        size = TYPE_SIZES.get(ntype, 22)

        # Ring highlight for head node
        border_colors = [
            '#f39c12' if n['node_id'] == head_id
            else '#fff' if n['is_selected']
            else '#aaa'
            for n in ns
        ]
        border_widths = [4 if n['node_id'] == head_id else 2 for n in ns]

        traces.append(go.Scatter(
            x=xs, y=ys,
            mode='markers+text',
            marker=dict(
                symbol=symbol, size=size, color=color,
                line=dict(color=border_colors, width=border_widths),
                opacity=[1.0 if n['is_selected'] else 0.45 for n in ns],
            ),
            text=labels,
            textposition='bottom center',
            textfont=dict(size=8, color='#2c3e50'),
            hovertext=hover, hoverinfo='text',
            name=ntype.capitalize(),
            legendgroup=ntype,
            showlegend=True,
        ))

    # --- Head annotation ---
    if head_id in pos:
        hx, hy = pos[head_id]
        traces.append(go.Scatter(
            x=[hx], y=[hy + 0.4],
            mode='text',
            text=['▼ HEAD'],
            textfont=dict(size=9, color='#f39c12'),
            showlegend=False, hoverinfo='none',
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=dict(text=title, font=dict(size=15)),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        plot_bgcolor='#f8f9fa',
        paper_bgcolor='white',
        legend=dict(orientation='h', y=-0.08, x=0.5, xanchor='center'),
        margin=dict(l=20, r=20, t=60, b=60),
        height=700,
    )
    return fig


fig = build_plotly_dag(g, title='Decision Graph — Two-Turn Design Session with Backtrack')
fig.show()
print('\nLegend: thick solid edge = active path | dashed = explored but not selected')
print('        gold ring = current HEAD | white border = selected | grey = deselected')


Legend: thick solid edge = active path | dashed = explored but not selected
        gold ring = current HEAD | white border = selected | grey = deselected


## Serialisation round-trip

The graph must survive JSON round-trip (for future Redis/DB storage).

In [7]:
d = g.to_dict()
g2 = DecisionGraph.from_dict(d)
d2 = g2.to_dict()

assert g2.node_count() == g.node_count(), 'node count mismatch after round-trip'
assert d2['head'] == d['head'],           'head mismatch after round-trip'
assert len(d2['edges']) == len(d['edges']), 'edge count mismatch'
print(f'Round-trip OK — {g2.node_count()} nodes, {len(d2["edges"])} edges')
print(f'head: {d2["head"][:8]}...')

Round-trip OK — 20 nodes, 19 edges
head: de2480d7...


## Step-by-step replay animation

Simulates how the graph grows node-by-node as events stream in,  
rendered as a **Plotly slider** (one frame per node added).

In [ ]:
# Replay: build sub-graphs of size 1..N and snapshot each
all_nodes = g.to_dict()['nodes']  # ordered by insertion
snapshots = []
for k in range(1, len(all_nodes) + 1):
    subset = all_nodes[:k]
    subset_ids = {n['node_id'] for n in subset}
    edges = [e for e in g.to_dict()['edges']
             if e['source'] in subset_ids and e['target'] in subset_ids]
    snapshots.append({'nodes': subset, 'edges': edges,
                      'head': subset[-1]['node_id']})

def pos_for_snapshot(snap):
    # Use final layout but only show subset of nodes
    return compute_dag_layout(snap)

# Build frames
frames = []
# Pre-compute final positions so layout doesn't jump
final_pos = compute_dag_layout(g.to_dict())

for snap in snapshots:
    snap_ids = {n['node_id'] for n in snap['nodes']}
    node_x, node_y, node_text, node_color, node_size = [], [], [], [], []
    for n in snap['nodes']:
        if n['node_id'] not in final_pos:
            continue
        x, y = final_pos[n['node_id']]
        node_x.append(x)
        node_y.append(y)
        node_text.append(n['label'][:25])
        node_color.append(TYPE_COLORS.get(n['type'], '#7f8c8d'))
        node_size.append(TYPE_SIZES.get(n['type'], 22))

    edge_x, edge_y = [], []
    for e in snap['edges']:
        if e['source'] in final_pos and e['target'] in final_pos:
            x0, y0 = final_pos[e['source']]
            x1, y1 = final_pos[e['target']]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]

    frames.append(go.Frame(
        data=[
            go.Scatter(x=edge_x, y=edge_y, mode='lines',
                       line=dict(color='#2c3e50', width=2), hoverinfo='none'),
            go.Scatter(x=node_x, y=node_y, mode='markers+text',
                       marker=dict(color=node_color, size=node_size,
                                   line=dict(color='white', width=2)),
                       text=node_text, textposition='bottom center',
                       textfont=dict(size=7)),
        ],
        name=str(len(snap['nodes'])),
    ))

# Initial frame
first = snapshots[0]
n0 = first['nodes'][0]
x0_init, y0_init = final_pos.get(n0['node_id'], (0, 0))
anim_fig = go.Figure(
    data=[
        go.Scatter(x=[], y=[], mode='lines', line=dict(color='#2c3e50', width=2)),
        go.Scatter(x=[x0_init], y=[y0_init], mode='markers+text',
                   marker=dict(color=[TYPE_COLORS.get(n0['type'], '#7f8c8d')],
                               size=[TYPE_SIZES.get(n0['type'], 22)],
                               line=dict(color='white', width=2)),
                   text=[n0['label'][:25]], textposition='bottom center',
                   textfont=dict(size=7)),
    ],
    frames=frames,
)
anim_fig.update_layout(
    title='Decision graph — step-by-step replay',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='#f8f9fa', height=600,
    updatemenus=[dict(
        type='buttons', showactive=False, y=0, x=0.5, xanchor='center',
        buttons=[
            dict(label='Play', method='animate',
                 args=[None, dict(frame=dict(duration=400, redraw=True),
                                  fromcurrent=True)]),
            dict(label='Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0), mode='immediate')]),
        ],
    )], 
    sliders=[dict(
        steps=[dict(method='animate', args=[[f.name], dict(mode='immediate')],
                    label=f.name) for f in frames],
        x=0.1, len=0.8, y=0,
        currentvalue=dict(prefix='Nodes added: ', visible=True),
    )],
)
anim_fig.show()

: 

## Notes for the UI implementation

The backend returns `{nodes, edges, head}` from `GET /sessions/{id}/decisions`.

### React Flow node data mapping

```js
// Convert backend nodes → React Flow nodes
const rfNodes = nodes.map(n => ({
  id: n.node_id,
  type: n.type,           // use as custom node component key
  data: {
    label: n.label,
    payload: n.payload,
    isSelected: n.is_selected,
    isHead: n.node_id === head,
  },
  position: { x: 0, y: 0 }, // let dagre/elkjs compute layout
}))

// Edges — already have source/target
const rfEdges = edges.map(e => ({
  ...e,
  animated: isOnActivePath(e.source, e.target),
  style: { stroke: isOnActivePath(e.source, e.target) ? '#1a252f' : '#bdc3c7' },
}))
```

### User selects a Pareto option

```js
// POST /sessions/{id}/decisions/{node_id}/select
const res = await fetch(`/sessions/${sessionId}/decisions/${nodeId}/select`, {
  method: 'POST',
  body: JSON.stringify({ reason: 'User clicked option in explorer' }),
})
const { select_node, graph_head } = await res.json()
// Re-fetch graph and update React Flow state
```

### SSE decision events during chat

```js
eventSource.addEventListener('decision', (e) => {
  const node = JSON.parse(e.data)  // { node_id, type, label, parent_id, is_selected }
  addNodeToGraph(node)             // incrementally update the graph view
})
```